# Predicting Employee Attrition and Identifying Flight Risk Factors

**Group Members:** Shania Siew (816039282), Terrence Murray (816038951), Tyler Baksh (816039328), Syam Manchikanti (816041877)

## Introduction

Employee attrition poses a significant and costly challenge for organizations across industries. When employees leave unexpectedly, companies face operational disruptions, diminished productivity, and substantial recruitment and training expenses, often amounting to 50–200% of the departing employee's annual salary. Despite these high stakes, many organizations continue to manage turnover reactively, addressing departures only after they occur rather than anticipating and preventing them.

This project develops a workforce intelligence platform that shifts attrition management from reactive to proactive. By leveraging machine learning classification models and survival analysis techniques, we aim to predict which employees are at risk of leaving and estimate the time horizon of their departure, providing organizations with a 3–6 month intervention window. Beyond prediction, we investigate the key factors driving flight risk — such as overtime, tenure, job satisfaction, and compensation — and assess whether the models treat employees fairly across protected attributes including gender, age, marital status, and education field.

Our analysis is built on the **IBM HR Analytics Employee Attrition & Performance** dataset, which provides detailed employee records encompassing tenure, salary, role, performance ratings, and attrition status. This dataset is well-suited for both classification and survival analysis tasks. We complement it with the **HR Analytics Job Change of Data Scientists** dataset to capture additional context around workforce mobility and career transitions.

The expected deliverables include a predictive attrition model with flight risk tier rankings, identification of primary attrition drivers through feature importance analysis (Random Forest, LASSO, SHAP), fairness audits across demographic subgroups, an ROI calculator estimating the cost savings of targeted retention interventions, and an interactive dashboard presenting survival curves, risk profiles, and department-level attrition trends.

## Part 1: Data Exploration and Cleaning

Before building any predictive models, we must first understand the structure, quality, and characteristics of our dataset. This step involves:

1. **Loading and inspecting** the IBM HR Analytics dataset to understand its schema, data types, and dimensions.
2. **Summary statistics** — computing descriptive statistics for numerical features and frequency distributions for categorical features to identify patterns and potential anomalies.
3. **Missing value analysis** — checking for null or missing entries across all columns and determining an appropriate handling strategy (imputation or removal).
4. **Duplicate detection** — identifying and removing any duplicate records that could bias the analysis.
5. **Irrelevant feature removal** — dropping columns that carry no predictive value (e.g., constant-value columns like `EmployeeCount`, `StandardHours`, and `Over18`, as well as identifiers like `EmployeeNumber`).
6. **Target variable inspection** — examining the distribution of the `Attrition` column to quantify class imbalance, which will inform our modeling and evaluation strategy.
7. **Data type encoding** — converting categorical variables (e.g., `Attrition`, `OverTime`, `Gender`) into numerical representations suitable for machine learning models.
8. **Outlier assessment** — using visualizations (box plots, histograms) to detect extreme values in features such as `MonthlyIncome`, `YearsAtCompany`, and `TotalWorkingYears`.

### 1.1.1 Loading and Inspecting the Dataset

We begin by downloading and loading the IBM HR Analytics Employee Attrition & Performance dataset. After loading, we inspect its shape, column names, and data types to gain an initial understanding of the features available for analysis.

In [ ]:
import polars as pl
import requests
import os

# Data Directory
DATA_DIR = "data/raw"
FILE_NAME = "WA_Fn-UseC_-HR-Employee-Attrition.csv"
os.makedirs(DATA_DIR, exist_ok=True)

# Download the dataset from source
url: str = "https://www.kaggle.com/api/v1/datasets/download/pavansubhasht/ibm-hr-analytics-attrition-dataset"

# Check if the dataset already exists
dataset_path = os.path.join(DATA_DIR, "ibm_hr_analytics_attrition_dataset.zip")
if os.path.exists(dataset_path):
    print("Dataset already exists. Skipping download.")
else:
    # Stream download the dataset
    response = requests.get(url, stream=True)
    if response.status_code == 200:
        with open(os.path.join(DATA_DIR, "ibm_hr_analytics_attrition_dataset.zip"), "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
    else:    print(f"Failed to download dataset. Status code: {response.status_code}")

# Unzip the dataset
import zipfile
with zipfile.ZipFile(dataset_path, 'r') as zip_ref:
    zip_ref.extractall(DATA_DIR)

Dataset already exists. Skipping download.


In [7]:
# Load the dataset using Polars
df = pl.read_csv(os.path.join(DATA_DIR, FILE_NAME))

# Display the first few rows of the dataset
print(df.head())

# List all columns with their data types
for col in df.columns:
    print(f"{col:30s} {str(df[col].dtype)}")

shape: (5, 35)
┌─────┬───────────┬────────────┬───────────┬───┬────────────┬────────────┬────────────┬────────────┐
│ Age ┆ Attrition ┆ BusinessTr ┆ DailyRate ┆ … ┆ YearsAtCom ┆ YearsInCur ┆ YearsSince ┆ YearsWithC │
│ --- ┆ ---       ┆ avel       ┆ ---       ┆   ┆ pany       ┆ rentRole   ┆ LastPromot ┆ urrManager │
│ i64 ┆ str       ┆ ---        ┆ i64       ┆   ┆ ---        ┆ ---        ┆ ion        ┆ ---        │
│     ┆           ┆ str        ┆           ┆   ┆ i64        ┆ i64        ┆ ---        ┆ i64        │
│     ┆           ┆            ┆           ┆   ┆            ┆            ┆ i64        ┆            │
╞═════╪═══════════╪════════════╪═══════════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ 41  ┆ Yes       ┆ Travel_Rar ┆ 1102      ┆ … ┆ 6          ┆ 4          ┆ 0          ┆ 5          │
│     ┆           ┆ ely        ┆           ┆   ┆            ┆            ┆            ┆            │
│ 49  ┆ No        ┆ Travel_Fre ┆ 279       ┆ … ┆ 10         ┆ 7          ┆ 1

### 1.1.2 Removing Irrelevant Features

Upon inspection, `EmployeeNumber` is a unique row identifier with no predictive value, and `EmployeeCount` is a constant column (every row equals 1). Neither contributes meaningful information for modeling attrition, so we drop them to reduce noise in the dataset.

In [ ]:
drop_cols: list[str] = ["EmployeeCount", "EmployeeNumber"]

# Drop the specified columns from the DataFrame
df = df.drop(drop_cols)